> **中文阅读版**：本文件是 `04_research_agent.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 研究智能体

**Docker 镜像**：`ml4t`

前三个 notebook 分别构建了各个组件：一个模型客户端、一个带截止时间和来源策略的搜索工具，以及一个带有门控的状态记录。本 notebook 将它们组装成一个对象：输入一个问题，返回一个概率，并附带其背后的一切；这也是本章后续每个 notebook 所操作的基本单元。

**学习目标**：
- 将模型客户端、搜索工具和轮次预算组装成一个端到端回答问题的智能体
- 解析并校验以代码围栏包裹、带前缀或形状错误的模型回复，并将其退回修正而不是直接失败
- 在代码中从预测推导置信度、情绪、证据类别和关键发现，使每个智能体的元数据都以相同方式生成
- 区分已给出概率承诺的智能体与耗尽轮次的智能体，并将后者排除在任何平均值之外
- 读取保存的运行记录，并在无需 API 调用的情况下复现数月前做出的预测

**书中参考**：第 24 章，24.6 节（设计流水线核心的研究智能体）

**前置条件**：[`01_react_reasoning`](01_react_reasoning.ipynb)（提供方）、[`02_tool_contracts`](02_tool_contracts.ipynb)（工具）、[`03_state_and_memory`](03_state_and_memory.ipynb)（状态与门控）。

In [2]:
"""The Research Agent - ReAct loop with rich output extraction."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import json
import re
from datetime import date

import polars as pl
from agent_fixtures import get_chapter_contested_question
from agent_observability import (
    TRACES_DIR,
    RunTrace,
    merge_calls,
    show_agent_timeline,
    trace_llm,
)
from agent_pipeline import validate_probabilities
from agent_providers import ChatMessage, LLMClient, TokenUsage, create_llm_client
from agent_schemas import (
    AgentForecastArtifact,
    AgentTrace,
    EvidenceQuality,
    ForecastQuestion,
    SearchResult,
    Sentiment,
)
from agent_tools import (
    MockSearchClient,
    SearchClient,
    ToolExecutor,
    create_search_client,
    eligible_results,
    format_search_results,
)

In [3]:
# RUN_LIVE=False (the default) replays the pinned 2026-06-09 trace named below:
# the notebook reloads that saved run and makes no API calls, so the agent
# outputs are stable and match the chapter. Set RUN_LIVE=True (with API keys) to
# forecast a current question live; that path produces different numbers.
RUN_LIVE = False
PINNED_TRACE = "04_research_agent_20260609T141730Z_b694ab4d0453.json"

LLM_PROVIDER = ""  # Blank reads .env; "mock" selects a synthetic fixture
MAX_STEPS = 5
MAX_SEARCH_RESULTS = 5

## 提示模板

系统提示固定了智能体的角色及其两条硬性约束：输出有效 JSON，且不得去寻找市场自身的价格。第二条对评估很重要。读取自己正被用来评分的预测市场的智能体是在抄答案，而抄答案会得到高分，因此该约束在提示中告知模型，并在下文通过保存的提示进行验证，而不是直接信任。

In [4]:
AGENT_SYSTEM_PROMPT = """\
You are a forecasting agent in a multi-agent forecasting system.

Your job:
1) Gather evidence by issuing web/news search queries when needed.
2) Then produce a binary probability forecast for the question.

You must follow the action schema exactly and output valid JSON only.
You must not browse prediction market prices unless they are explicitly provided.
Cite a retained source URL or title in your rationale. If evidence is insufficient,
return {"action":"abstain","rationale":"what is missing"}."""

### 步骤提示

提供问题、可选的市场上下文以及动作 schema。
智能体每一步必须恰好输出一个 JSON 动作。

In [5]:
def build_step_prompt(
    question: ForecastQuestion,
    market_price: float | None = None,
    max_steps: int | None = None,
) -> str:
    """Format the step prompt with question context.

    When ``max_steps`` is given, the agent is told its search budget and is
    nudged to commit once it has enough evidence - searching is not free, and an
    agent that knows its budget stops reformulating the same query and forecasts.
    """
    prompt = f"QUESTION:\n{question.question}\n\n"
    if question.description:
        prompt += f"MARKET CONTEXT:\n{question.description}\n\n"
    if market_price is not None:
        prompt += f"MARKET IMPLIED PROBABILITY (p_yes):\n{market_price}\n\n"
    if max_steps is not None:
        prompt += (
            f"SEARCH BUDGET: you have at most {max_steps} steps. Search only when a "
            "specific, named fact is missing - do not reformulate a query you already "
            "ran. As soon as you have a base rate and the current signal, forecast.\n\n"
        )
    prompt += (
        "NEXT ACTION SCHEMA (output JSON only):\n"
        'If you need more info:\n{"action":"search","query":"..."}\n'
        "If you are ready to forecast:\n"
        '{"action":"forecast","p_yes":0.XX,"rationale":"short explanation '
        'grounded in evidence and base rates"}\n\n'
        "Pick exactly one action."
    )
    return prompt

## JSON 解析

LLM 有时会将 JSON 包裹在 Markdown 代码块中，或添加尾随文本。
稳健解析会先尝试 `json.loads`，然后回退到提取第一个 `{...}` 块。

In [6]:
def parse_json(raw: str) -> dict:
    """Parse JSON from LLM output with fallback bracket extraction."""
    raw = raw.strip()
    try:
        parsed = json.loads(raw)
        return (
            parsed
            if isinstance(parsed, dict)
            else {"action": "invalid", "error": "Expected JSON object"}
        )
    except json.JSONDecodeError:
        pass

    # Strip markdown code fences
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    try:
        parsed = json.loads(raw)
        return (
            parsed
            if isinstance(parsed, dict)
            else {"action": "invalid", "error": "Expected JSON object"}
        )
    except json.JSONDecodeError:
        pass

    # Extract first {...} block
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            return (
                parsed
                if isinstance(parsed, dict)
                else {"action": "invalid", "error": "Expected JSON object"}
            )
        except json.JSONDecodeError:
            pass

    return {"action": "invalid", "error": "JSON parse failure"}

## 丰富输出提取

智能体生成预测后，我们从原始 LLM 输出中提取额外元数据。这些字段用置信度、情绪、关键发现、不确定性和证据质量来充实 `AgentForecastArtifact`。

### 置信度提取

如果预测 JSON 包含 `confidence` 字段，我们直接使用它（截断到 $[0, 1]$）。否则，我们根据概率的极端程度推断置信度：$\text{confidence} = 2 \cdot |p_{\text{yes}} - 0.5|$。

In [7]:
def extract_confidence(action: dict) -> float:
    """Extract confidence from action dict or infer from probability."""
    if "confidence" in action:
        try:
            return max(0.0, min(1.0, float(action["confidence"])))
        except (TypeError, ValueError):
            pass
    p = float(action.get("p_yes", 0.5))
    return round(abs(p - 0.5) * 2, 3)

## 解析并校验模型回复

被要求输出 JSON 的模型大多数时候确实会输出 JSON。其余时候，它会把对象包裹在 Markdown 代码围栏中，在对象前添加一句解释，或返回格式良好但并非动作的内容。这三种情况都很常见，都不是错误条件，而遇到它们就崩溃的智能体是无法无人值守运行的智能体。

`parse_json` 只接受一个 JSON 对象，无论是否带围栏，并将其他任何内容转换为显式的解析失败动作。随后 `validate_action` 按两个已记录的 schema 检查该对象：搜索需要非空字符串查询，预测需要有限的数值 `p_yes` 和字符串形式的理由，且概率在用于任何操作前会被截断到 $[0, 1]$。未通过任一检查的回复会消耗一个轮次，并被退回给模型进行修正。

两者都来自 `agent_research.py`，而不是在这里编写，因为本章的每个智能体都必须以相同方式拒绝相同的形状：不同 notebook 之间解析器不一致，会成为看似判断分歧的分歧来源。

## 派生字段

模型返回一个概率和一个理由。产物上的其他所有内容都由下面的函数根据这两者计算得出。将推导保留在代码中而不是向模型索取，意味着每个智能体的元数据都以相同方式生成，这正是使智能体可比较的原因；这也意味着这些字段的好坏完全取决于其定义，而其中若干定义是刻意粗糙的。凡是在打印这些字段的地方，都会将其标记为启发式方法。

### 置信度

当模型主动提供 `confidence` 字段时，就使用它，并将其截断到 $[0, 1]$。否则，置信度取为与均等概率的距离，$\text{confidence} = 2\,\lvert p_{\text{yes}} - \tfrac{1}{2} \rvert$，该值在均等概率处为零，在任一确定性处为一。

这是关于概率所处位置的陈述，而不是关于其背后证据的陈述。一个读了四十份文档并得出该问题确实势均力敌结论的智能体得分为零，一个什么都没读、以均等概率猜测的智能体也同样得分为零。这个数字很廉价，而且对每个智能体含义相同，这正是它的用途。真正能回答这个问题的是校准：以百分之七十表述的概率是否约在百分之七十的情况下应验。[`09_evaluation_and_governance`](09_evaluation_and_governance.ipynb) 构建了那套计算，并在合成输入上运行它，因为要真实测量它，需要在问题解决前记录预测，而本章没有任何捕获记录具备这一点。

### 情绪提取

将 $p_{\text{yes}}$ 机械地映射到五级情绪量表。

In [8]:
def extract_sentiment(p_yes: float) -> Sentiment:
    """Infer sentiment from probability."""
    if p_yes > 0.8:
        return Sentiment.STRONGLY_BULLISH
    if p_yes > 0.6:
        return Sentiment.BULLISH
    if p_yes > 0.4:
        return Sentiment.NEUTRAL
    if p_yes > 0.2:
        return Sentiment.BEARISH
    return Sentiment.STRONGLY_BEARISH

### 关键发现

当模型被要求给出简短理由（rationale）时，它通常会列举自己的原因，并自行选择格式：有时每行一个条目，前面带破折号或编号；有时则连在一句话里写成 `(1) ... (2) ...`。该函数同时接受两种形式，优先选择存在的行首条目，若没有则回退到行内标记，并且要求至少两个标记，以免普通文本中单独一个带括号的数字被当作列表。

没有列举任何内容的理由不会产出任何结果，这是诚实的结果而非失败：该字段记录的是模型选择如何呈现其理由，而不是它有多少条理由。

最后一个行内条目需要小心处理，因为最终标记之后的文本会一直延续到模型接下来写的内容，所以该条目必须结束在它自己的句子边界处。找到这个边界意味着要区分句号与缩写：*U.S. inflation remains elevated* 是一个句子，而若规则在每个句点处截断，就会返回 `U.S`。只有当前一个句点前的字母不是大写、且下一个词以大写字母开头时，该条目才结束。

In [9]:
def extract_key_findings(rationale: str) -> list[str]:
    """Extract bullet points and numbered items from rationale."""
    findings = []
    for line in rationale.split("\n"):
        line = line.strip()
        if re.match(r"^[-•*]\s+", line):
            findings.append(re.sub(r"^[-•*]\s+", "", line).strip())
        elif re.match(r"^\d+[.)]\s+", line):
            findings.append(re.sub(r"^\d+[.)]\s+", "", line).strip())
    return findings[:10]

### 不确定性提取

识别理由（rationale）中包含不确定性措辞的句子。

In [10]:
def extract_uncertainties(rationale: str) -> list[str]:
    """Identify sentences mentioning uncertainty."""
    uncertainty_words = {
        "uncertain",
        "unclear",
        "unknown",
        "risk",
        "caveat",
        "however",
        "but",
        "although",
    }
    sentences = re.split(r"[.!?]+", rationale)
    return [
        s.strip() for s in sentences if s.strip() and any(w in s.lower() for w in uncertainty_words)
    ][:5]

### 证据质量评估

这种基于数量的启发式方法只使用查询次数和结果数量。它不验证来源可信度、独立性或 Point-in-Time (PIT) 可用性。

In [11]:
def assess_evidence_quality(sources_consulted: int, queries_made: int) -> EvidenceQuality:
    """Assess evidence quality from search metrics."""
    if sources_consulted >= 10 and queries_made >= 3:
        return EvidenceQuality.HIGH
    if sources_consulted >= 5 or queries_made >= 2:
        return EvidenceQuality.MEDIUM
    return EvidenceQuality.LOW

## ReAct 步骤处理器

ReAct 循环在每一步处理三种动作类型之一：搜索、预测或无法识别的动作。将每个分支提取为一个小型自由函数，可使智能体类专注于迭代和组装。

In [12]:
# Check the returned sources, including supplied context.

## ResearchAgent 类

该智能体持有 LLM、搜索执行器和迭代预算。其 `run()` 方法遍历 ReAct 循环，将每一步分派给上面的处理器，并返回一个带有累积元数据的 `AgentForecastArtifact`。该类在 NB06（多智能体）和 NB08（完整流水线）中原样复用。

In [13]:
class ResearchAgent:
    """ReAct research agent producing evidence-linked probability forecasts.

    Implements the same loop as the AIA Forecaster's research agent:
    search for evidence, then forecast with rich metadata.
    """

    def __init__(
        self,
        llm: LLMClient,
        search: SearchClient | None = None,
        agent_id: str = "agent_0",
        max_steps: int = 6,
        max_search_results: int = 5,
    ) -> None:
        self.llm = llm
        self.executor = ToolExecutor(search=search)
        self.agent_id = agent_id
        self.max_steps = max_steps
        self.max_search_results = max_search_results

    def run(
        self,
        question: ForecastQuestion,
        market_price: float | None = None,
        evidence: list[SearchResult] | None = None,
    ) -> AgentForecastArtifact:
        """Research within the budget; retain unsupported attempts as abstentions."""
        if self.max_steps < 1 or self.max_search_results < 1:
            raise ValueError("Research budgets must be positive")
        cutoff = date.fromisoformat(question.cutoff_date) if question.cutoff_date else None
        messages = [
            ChatMessage(role="system", content=AGENT_SYSTEM_PROMPT),
            ChatMessage(
                role="user", content=build_step_prompt(question, market_price, self.max_steps)
            ),
        ]
        traces: list[AgentTrace] = []
        sources = eligible_results(evidence or [], cutoff)
        if sources:
            traces.append(AgentTrace(step=0, action="supplied_evidence", results=sources))
            messages.append(ChatMessage(role="user", content=format_search_results(sources)))
        total_tokens = TokenUsage()
        queries_made = 0
        p_yes = None
        rationale = ""
        raw_action = {}
        reason = "Budget exhausted without a supported forecast"
        status = "abstained"

        for step in range(1, self.max_steps + 2):
            synthesis_only = step > self.max_steps
            if synthesis_only:
                if not sources:
                    break
                messages.append(
                    ChatMessage(
                        role="user",
                        content=(
                            "Search budget exhausted. Use only the retained evidence to forecast, "
                            "citing its URL or title. If it is insufficient, abstain. No further search."
                        ),
                    )
                )
            try:
                response, usage = self.llm.complete_with_usage(messages, json_mode=True)
            except Exception as exc:
                reason = f"Model call failed ({type(exc).__name__})"
                status = "failed"
                traces.append(AgentTrace(step=step, action="provider_error", llm_raw=reason))
                break
            total_tokens = total_tokens + usage
            messages.append(ChatMessage(role="assistant", content=response))
            action = parse_json(response)
            action_type = action.get("action", "invalid")
            if action_type == "search" and not synthesis_only:
                query = action.get("query", "")
                results = self.executor.execute_search(query, self.max_search_results, cutoff)
                log = self.executor.execution_log[-1]
                queries_made += int(log.status in {"success", "error"})
                sources.extend(results)
                traces.append(
                    AgentTrace(
                        step=step, action="search", query=query, results=results, llm_raw=response
                    )
                )
                messages.append(
                    ChatMessage(
                        role="tool",
                        content=(
                            format_search_results(results)
                            if results
                            else f"No eligible evidence. Search status: {log.status}. {log.result_preview}"
                        ),
                    )
                )
                continue
            traces.append(AgentTrace(step=step, action=str(action_type), llm_raw=response))
            if action_type == "abstain":
                reason = str(action.get("rationale") or "Model reported insufficient evidence")
                break
            if action_type == "forecast":
                try:
                    if isinstance(action["p_yes"], bool):
                        raise ValueError("Probability must be a number, not a boolean")
                    candidate = float(action["p_yes"])
                    validate_probabilities([candidate])
                except (KeyError, TypeError, ValueError):
                    reason = "Invalid probability; forecast rejected"
                else:
                    explanation = action.get("rationale")
                    if not sources:
                        reason = "No eligible evidence; forecast rejected"
                    elif not isinstance(explanation, str) or not explanation.strip():
                        reason = "Missing rationale; forecast rejected"
                    elif not any(
                        (r.url and r.url in explanation) or (r.title and r.title in explanation)
                        for r in sources
                    ):
                        reason = "Rationale must identify a retained source URL or title"
                    else:
                        p_yes, rationale, raw_action = candidate, explanation, action
                        status = "accepted"
                        break
            else:
                reason = "Invalid action or search requested after the budget was exhausted"
            messages.append(ChatMessage(role="tool", content=reason))

        return AgentForecastArtifact(
            agent_id=self.agent_id,
            p_yes=p_yes,
            rationale=rationale,
            traces=traces,
            status=status,
            failure_reason=None if status == "accepted" else reason,
            messages=[{"role": m.role, "content": m.content} for m in messages],
            execution_mode=(
                "synthetic"
                if isinstance(self.executor.search, MockSearchClient)
                or any(
                    label in getattr(self.llm, "model_name", "").lower()
                    for label in ("mock", "synthetic")
                )
                else "live"
            ),
            confidence=extract_confidence(raw_action) if p_yes is not None else 0.0,
            sentiment=extract_sentiment(p_yes) if p_yes is not None else Sentiment.NEUTRAL,
            key_findings=extract_key_findings(rationale),
            evidence_quality=assess_evidence_quality(len(sources), queries_made),
            uncertainties=extract_uncertainties(rationale),
            token_usage=total_tokens,
            search_queries_made=queries_made,
            sources_consulted=len(sources),
        )

该问题来自 `agent_fixtures.py` 中的 `CHAPTER_CONTESTED_QUESTION`：*“美联储会在 2026 年加息吗？”*。它之所以有争议，正是在这里要紧的那种特定意义上：同一个智能体的两次运行会就此得出相反结论，因此本章凡是以分歧为主题之处都使用这个问题。[`07_adversarial_debate`](07_adversarial_debate.ipynb) 让多个智能体就此争论，[`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb) 则在其上运行完整流水线。[`06_multi_agent_research`](06_multi_agent_research.ipynb) 则改用配套的 `CHAPTER_CLEAR_QUESTION`，在该问题上各智能体意见一致，因此可以将两者进行比较。

在默认路径下，notebook 会重放固定的捕获记录（pinned capture），而不是调用任何服务；下面的单元格报告提供商、搜索工具以及它所读取的捕获记录日期，每一项都取自已保存记录。设置 `RUN_LIVE = True` 并提供 `ANTHROPIC_API_KEY` 与 `TAVILY_API_KEY`，则会改为预测当前问题，并且不会复现下面的数值。

In [14]:
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    search = create_search_client()

    # Wrap the client in a TracingLLMClient so every prompt and raw response is
    # captured for the run trace we persist at the end of the notebook.
    tracer_0 = trace_llm(llm, label="agent_0")
    agent = ResearchAgent(
        llm=tracer_0,
        search=search,
        agent_id="agent_0",
        max_steps=MAX_STEPS,
        max_search_results=MAX_SEARCH_RESULTS,
    )

    question = get_chapter_contested_question()
    provider_name = llm.model_name
    search_name = type(search).__name__
    artifact = agent.run(question)
else:
    # Replay: reload the pinned trace and rehydrate its two saved agents. The
    # first is the single-agent forecast inspected here; the second is the
    # preview pair shown later. Display cells consume these unchanged.
    pinned_run = RunTrace.load(TRACES_DIR / PINNED_TRACE)
    question = pinned_run.question_obj()
    provider_name = pinned_run.provider
    search_name = "replay (pinned trace)"
    replayed_artifacts = pinned_run.agent_artifacts()
    artifact = replayed_artifacts[0]

print(f"Mode: {'LIVE' if RUN_LIVE else 'REPLAY (pinned 2026-06-09 trace)'}")
print(f"Provider: {provider_name}")
print(f"Search: {search_name}")
print(f"Question: {question.question}\n")

Mode: REPLAY (pinned 2026-06-09 trace)
Provider: claude-sonnet-4-20250514
Search: replay (pinned trace)
Question: Will the Federal Reserve hike rates in 2026?



## 检查预测产物

一个对象保存了此次运行产生的全部内容。前三个字段是模型实际给出的内容；其余字段由上面的提取函数根据这些内容计算得到，并且在任何打印它们的地方都标注为启发式，以免把某个派生类别误读为测量值。

In [15]:
print("=== Forecast ===")
print(f"Agent:       {artifact.agent_id}")
print(f"Status:      {artifact.status}")
print(f"p(YES):      {artifact.p_yes}")
if artifact.failure_reason:
    print(f"Reason:      {artifact.failure_reason}")
print(f"Confidence:  {artifact.confidence:.2f}")
print(f"Sentiment:   {artifact.sentiment.value}")
print(f"Evidence:    {artifact.evidence_quality.value}")
print(f"Queries:     {artifact.search_queries_made}")
print(f"Sources:     {artifact.sources_consulted}")
print(f"Tokens:      {artifact.token_usage.total_tokens:,}")

=== Forecast ===
Agent:       agent_0
Status:      legacy_capture
p(YES):      0.65
Confidence:  0.30
Sentiment:   bullish
Evidence:    high
Queries:     4
Sources:     20
Tokens:      23,335


In [16]:
print(f"\nRationale:\n{artifact.rationale[:400]}")


Rationale:
Based on current evidence, there's a moderate-to-high probability of Fed rate hikes in 2026. Key factors: (1) Markets now price >50% chance of rate hikes by end-2026 due to inflation concerns from Iran war and rising oil prices ($110+ crude); (2) Fed currently holds rates at 4.25-4.5% after cutting 1.75% in 2024-2025; (3) Inflation pressures mounting from energy costs and tariff impacts; (4) Histo


In [17]:
if artifact.key_findings:
    print("\nKey Findings:")
    for f in artifact.key_findings:
        print(f"  • {f}")

if artifact.uncertainties:
    print("\nUncertainties:")
    for u in artifact.uncertainties:
        print(f"  • {u}")


Uncertainties:
  • However, uncertainty remains around economic growth, labor market conditions, and geopolitical developments that could shift policy direction


## 执行轨迹

每个搜索查询及其结果都会记录在轨迹中。来自 `agent_observability` 的 `show_agent_timeline` 会按顺序渲染整个运行过程：每个查询、它返回的文档（标题、日期、URL 和摘要），以及带有未截断理由（rationale）的预测。这是每个智能体的可观测性视图，会在后续多智能体 notebook 中复用。

In [18]:
print(show_agent_timeline(artifact))

  AGENT_0  ·  p_yes=0.65  status=legacy_capture  confidence=0.30  sentiment=bullish  evidence=high
  searches=4  sources=20  tokens=23,335
  Step 1 · SEARCH  "Federal Reserve interest rates 2024 2025 current policy outlook"
      1. [?] Federal Reserve keeps interest rates steady as inflation uncertainty ...
         https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-interest-rate.html
         The Fed shifted as inflation slowed, cutting its target interest rate
         by 1.75% through 2024 and 2025. Oil prices have increased more than
         60%
      2. [?] United States Interest Rate Outlook & Forecast - FocusEconomics
         https://www.focus-economics.com/country-indicator/united-states/interest-rate
         The federal funds target rate ended 2024 at 4.50%, compared to the
         end-2023 value of 5.50% and the figure a decade earlier of 0.25%. It
         averaged 2.02% over the
      3. [?] Federal Reserve Cuts Interest Rates in Late 2

## 工具执行审计

上面的时间线由 agent 自身的轨迹汇编而成，这是模型对本次运行的叙述。`ToolExecutor` 保留第二份独立记录，说明它实际被要求做什么。在一次实时运行中，该日志会携带每次调用的 `duration_ms`；而固定轨迹不会持久化 wall-clock 计时，因为在一台机器上测得的时长无法在另一台机器上复现，所以回放路径改为从已保存的轨迹中重建查询、状态和结果计数。

用表格而非对齐文本：同样的三个字段，以一种可排序、过滤和连接而无需任何人解析字符串的形式呈现。

In [19]:
if RUN_LIVE:
    audit_df = (
        pl.DataFrame(
            [
                {
                    "query": entry.args.get("query", "?"),
                    "status": entry.status,
                    "duration_ms": round(entry.duration_ms, 1),
                }
                for entry in agent.executor.execution_log
            ]
        )
        if agent.executor.execution_log
        else pl.DataFrame({"query": [], "status": [], "duration_ms": []})
    )
else:
    # Replay: reconstruct the search audit from the saved agent traces.
    search_steps = [t for t in artifact.traces if t.action == "search"]
    audit_df = (
        pl.DataFrame(
            [
                {
                    "query": t.query or "?",
                    "status": "ok",
                    "results": len(t.results),
                }
                for t in search_steps
            ]
        )
        if search_steps
        else pl.DataFrame({"query": [], "status": [], "results": []})
    )
audit_df

query,status,results
str,str,i64
"""Federal Reserve interest rates…","""ok""",5
"""Federal Reserve 2026 interest …","""ok""",5
"""Federal Reserve current intere…","""ok""",5
"""Federal Reserve rate hike prob…","""ok""",5


新鲜度检查通过，一致性报告指出没有需要执行的截止日期，这是关于问题的陈述，而不是本次运行的缺陷。`CHAPTER_CONTESTED_QUESTION` 带有 `cutoff_date=""`，因为它在被捕获时仍是开放问题：它问的是 2026 年剩余时间，所以当天可获得的任何文档都不可能包含答案，也不需要排除任何日期。门控没有可检查的内容，才是正确结果。它对 `agent_fixtures.py` 中已解决的问题有约束力，每个问题都带有截止日期，而读取超过截止日期的内容，正是预测结果被伪造的方式。

覆盖率则是另一回事，它的失败更多说明的是 adapter，而不是 agent。上面每个条目都被标记为 `search_results`，因为轨迹记录的是 agent 搜索了什么，而不是它判定每个结果属于哪种证据。覆盖率门控要求一个 `base_rate` 条目，却找不到；无论 agent 研究得多么充分，它都不可能找到。把这种情况解读为“agent 跳过了基准率”，就是把重建的局限性误读为关于本次运行的发现。

这正是 [`03_state_and_memory`](03_state_and_memory.ipynb) 从另一个方向提出的论点：证据类型是在收集过程中做出的判断，而事后根据轨迹写成的记录无法恢复它。一个将接受覆盖率门控的 agent 必须边运行边写入其状态，在每条结果到达时对其进行分类，而不是等到最后从日志中推断分类。

## Agent 摘要格式

当多个 agent 并行运行时，它们的预测会被汇总给 supervisor 和 debate 阶段。下游使用的就是这种格式。

In [20]:
def format_agent_summary(a: AgentForecastArtifact) -> str:
    """Format an agent artifact as a summary for supervisor/debate prompts."""
    lines = [
        f"Agent: {a.agent_id}",
        f"Probability (p_yes): {a.p_yes}",
        f"Status: {a.status}",
        f"Confidence: {a.confidence:.2f}",
        f"Rationale: {a.rationale}",
    ]
    if a.key_findings:
        lines.append("Key findings:")
        for f in a.key_findings[:3]:
            lines.append(f"  - {f}")
    for trace in a.traces:
        for source in trace.results:
            lines.append(f"Source: {source.title} | {source.url}")
            lines.append(f"Excerpt: {(source.snippet or '')[:400]}")
    return "\n".join(lines)

In [21]:
print(format_agent_summary(artifact))

Agent: agent_0
Probability (p_yes): 0.65
Status: legacy_capture
Confidence: 0.30
Rationale: Based on current evidence, there's a moderate-to-high probability of Fed rate hikes in 2026. Key factors: (1) Markets now price >50% chance of rate hikes by end-2026 due to inflation concerns from Iran war and rising oil prices ($110+ crude); (2) Fed currently holds rates at 4.25-4.5% after cutting 1.75% in 2024-2025; (3) Inflation pressures mounting from energy costs and tariff impacts; (4) Historical precedent shows Fed responds to persistent inflation with rate increases. However, uncertainty remains around economic growth, labor market conditions, and geopolitical developments that could shift policy direction.
Source: Federal Reserve keeps interest rates steady as inflation uncertainty ... | https://www.usbank.com/investing/financial-perspectives/market-news/federal-reserve-interest-rate.html
Excerpt: The Fed shifted as inflation slowed, cutting its target interest rate by 1.75% through 202

## 两个 Agent，同一个问题

同一个类使用不同的 id 会得到第二个样本。其他方面没有任何差异：相同的问题、相同的提示词、相同的搜索工具、相同的模型。下面的两个概率仍然出现分化，这正是本章其余部分所基于的观察。这种变异来自哪里——每次搜索返回了哪些文档，以及模型自身的采样——在这里并未被识别，[`06_multi_agent_research`](06_multi_agent_research.ipynb) 也没有识别它；那个 notebook 所做的是运行足够多的副本，使这种离散程度值得关注。

In [22]:
if RUN_LIVE:
    tracer_1 = trace_llm(llm, label="agent_1")
    agent_b = ResearchAgent(llm=tracer_1, search=search, agent_id="agent_1", max_steps=MAX_STEPS)
    artifact_b = agent_b.run(question)
else:
    # Replay: the pinned trace's second saved agent is this preview run.
    artifact_b = replayed_artifacts[1]

for a in [artifact, artifact_b]:
    print(f"{a.agent_id}: status={a.status}, p_yes={a.p_yes}")
if artifact.p_yes is not None and artifact_b.p_yes is not None:
    print(f"Difference: {abs(artifact.p_yes - artifact_b.p_yes):.2f}")

agent_0: status=legacy_capture, p_yes=0.65
agent_1: status=legacy_capture, p_yes=0.25
Difference: 0.40


**解读**：被捕获的 agent 在搜索和加权证据的方式上不同。它们的概率差异可能反映检索、采样、提示词和模型偏差。这两次运行并未隔离这些效应，也未确立独立信息。Notebook 06 和 07 在其他问题上考察了一个 ensemble 和 debate；需要已解决且可比较的运行，才能评估二者中是否有任何一方能改进预测。

## 持久化运行轨迹

artifact 保存结构化预测；包裹在每个 agent 周围的 `TracingLLMClient` 保存原始对话。`RunTrace.capture` 将两者——问题、两个 agent 的 artifact，以及每一次 prompt/response——打包成 `forecast_traces/` 下的一个 JSON 记录，这与多 agent notebook（NB06–NB08）写入的是同一种可审计格式。使用 `RunTrace.load` 重新加载它，即可精确回放 agent 看到和说过的内容——这正是上面默认的 `RUN_LIVE = False` 路径所做的。实时运行会在这里写入一份新的轨迹；默认的回放运行会报告它加载的固定轨迹，而不是覆盖它。

In [23]:
if RUN_LIVE:
    run = RunTrace.capture(
        notebook="04_research_agent",
        provider=provider_name,
        question=question,
        params={"max_steps": MAX_STEPS, "max_search_results": MAX_SEARCH_RESULTS},
        agents=[artifact, artifact_b],
        notes="Single research agent plus a two-agent preview on the contested question.",
        llm_calls=merge_calls(tracer_0, tracer_1),
    )
    trace_path = run.save()
    print(
        f"Saved {len(run.llm_calls)} model calls "
        f"({run.total_tokens():,} tokens) → {trace_path.relative_to(trace_path.parents[1])}"
    )
else:
    # Replay: report the pinned trace we loaded rather than writing a new file.
    run = pinned_run
    trace_path = TRACES_DIR / PINNED_TRACE
    print(
        f"Replayed {len(run.llm_calls)} model calls "
        f"({run.total_tokens():,} tokens) from {trace_path.name}"
    )

Replayed 10 model calls (42,266 tokens) from 04_research_agent_20260609T141730Z_b694ab4d0453.json


## 关键要点

1. **ResearchAgent** 将 ReAct 循环与丰富的输出抽取相结合——每个预测都附带置信度、情绪、证据质量和不确定性
2. **结构化 artifact**：`AgentForecastArtifact` 捕获完整运行——概率、推理、轨迹和 token 用量
3. **稳健解析**：JSON 抽取能优雅处理 markdown 代码块和格式错误的 LLM 输出
4. **可复用**：这个类是多 agent（NB06）、debate（NB07）和完整流水线（NB08）notebook 的构建块
5. **Token 跟踪**：每次 LLM 调用都会被计量以进行成本分析

**下一步**：[`aggregation`](05_aggregation_math.ipynb)——组合多个概率估计的数学基础（Neyman 极值化、加权聚合）。

**本书**：第 24.6 节讨论 agent 设计模式，包括 agent 复杂度与预测校准之间的权衡。

### 失败测试：没有证据的概率

该脚本化模型故意在获取证据之前尝试预测。实际研究循环必须保留该尝试并弃权。不涉及任何实时模型或编造的表现结果。

In [24]:
class UnsupportedForecastFixture:
    model_name = "synthetic-failure-fixture"

    def complete_with_usage(self, messages, **kwargs):
        return '{"action":"forecast","p_yes":0.9,"rationale":"No sources consulted"}', TokenUsage()


unsupported = ResearchAgent(UnsupportedForecastFixture(), max_steps=1).run(
    ForecastQuestion("Deliberately unsupported teaching forecast")
)
assert unsupported.status == "abstained" and unsupported.p_yes is None
print(unsupported.status, unsupported.failure_reason)

abstained No eligible evidence; forecast rejected


## 对当前循环的全新验证

本次 2026 年 10 月 2 日的采集针对一个尚未解决的 10 月 FOMC 问题，使用了真实的 Sonnet 4.6 和 Tavily 调用。请将其保留的证据和模型 I/O 与上方原始 6 月采集一并检查。该预测没有结果评分；验收意味着有效的概率、理由以及符合条件且被引用的证据。

In [25]:
verified_run = RunTrace.load(TRACES_DIR / "verified_research_20261002T193656Z_f169a52a0918.json")
print(verified_run.question_obj().question)
print(f"Status: {verified_run.status}; p={verified_run.final_probability}")
print(f"Model calls: {len(verified_run.llm_calls)}; tokens: {verified_run.total_tokens():,}")
print(verified_run.agent_artifacts()[0].rationale)
print(show_agent_timeline(verified_run.agent_artifacts()[0]))

Will the Federal Reserve lower the upper bound of its federal funds target range at the October 27-28, 2026 FOMC meeting?
Status: accepted; p=0.07
Model calls: 4; tokens: 8,613
Based on evidence from https://www.cnbc.com/2026/09/16/fed-rate-decision-september-2026.html, the Fed hiked rates to 3.75%-4% at the September 2026 meeting, with 16 of 18 participants expecting another rate hike later in 2026. From https://streetstats.finance/rates/fedfunds, futures markets as of late September 2026 are pricing the rate to rise to ~4.2% by December 2026, indicating further tightening expected. The September 2026 FOMC projections (https://www.federalreserve.gov/monetarypolicy/fomcprojtabl20260916.htm) show a median federal funds rate of 4.1% for end-2026, consistent with one more hike. All signals point to continued tightening, making a rate cut (lower upper bound) at the October 27-28 meeting extremely unlikely. Probability of YES is approximately 7%.
  VERIFICATION_RESEARCH  ·  p_yes=0.07  stat